# Publication-Ready Visualization Dashboard

**Project**: Health Data Lake - Serverless FHIR Pipeline Performance Study  
**Date**: November 2025  
**Purpose**: Create publication-quality figures and interactive dashboards for results presentation

---

## Overview

This notebook creates:

1. **Publication Figures**: High-resolution plots for academic reports (PNG 300 DPI, SVG, PDF)
2. **Interactive Dashboards**: Plotly visualizations for web/HTML deployment
3. **Statistical Summaries**: APA-formatted tables and result summaries
4. **Performance Metrics**: Automated pipeline monitoring charts

All outputs are saved to `reports/figures/` and `reports/tables/` for reproducibility.

---

## 1. Setup and Configuration

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(12345)

# Configure matplotlib for publication quality
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['figure.titlesize'] = 14

# Set seaborn style
sns.set_style("whitegrid")
sns.set_palette("colorblind")

# Create output directories
pub_dir = Path('reports/figures/publication')
pub_dir.mkdir(parents=True, exist_ok=True)

interactive_dir = Path('reports/figures/interactive')
interactive_dir.mkdir(parents=True, exist_ok=True)

print("✓ Libraries imported and configuration complete")
print(f"✓ Publication figures directory: {pub_dir}")
print(f"✓ Interactive figures directory: {interactive_dir}")

## 2. Load Data and Results

In [ ]:
# Load processed data
data_path = Path('data/processed/combined_data.parquet')
df = pd.read_parquet(data_path)

# Load statistical results
results_path = Path('reports/statistical_results.csv')
if results_path.exists():
    results = pd.read_csv(results_path).iloc[0].to_dict()
else:
    print("⚠ Statistical results not found. Run 02_statistical_analysis.ipynb first.")
    results = {}

print(f"✓ Loaded {len(df):,} records")
print(f"✓ Loaded {len(results)} statistical results")

## 3. Figure 1: Executive Summary Dashboard (Matplotlib)

Four-panel summary of key findings for report frontmatter.

In [ ]:
# Create 2x2 executive summary figure
fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)

# Panel A: Latency comparison (box plot)
ax1 = fig.add_subplot(gs[0, 0])
manual_baseline = results.get('manual_baseline_sec', 1200)
data_for_box = [
    [manual_baseline] * len(df),  # Manual baseline (constant)
    df['latency_sec'].values  # Automated measured
]
bp = ax1.boxplot(data_for_box, labels=['Manual\n(Baseline)', 'Automated\n(Measured)'],
                 patch_artist=True, widths=0.6)
bp['boxes'][0].set_facecolor('lightcoral')
bp['boxes'][1].set_facecolor('lightgreen')
ax1.set_ylabel('Latency (seconds)', fontweight='bold')
ax1.set_title('A. Processing Speed Comparison', fontweight='bold', pad=10)
ax1.grid(True, alpha=0.3, axis='y')

# Add speedup annotation
if 'speedup_factor' in results:
    speedup = results['speedup_factor']
    ax1.text(0.5, 0.95, f'{speedup:.1f}x Faster', 
             transform=ax1.transAxes, ha='center', va='top',
             fontsize=14, fontweight='bold', color='green',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Panel B: Validation accuracy (bar chart)
ax2 = fig.add_subplot(gs[0, 1])
manual_val_rate = results.get('manual_validation_rate', 0.85) * 100
auto_val_rate = results.get('automated_validation_rate', 0.95) * 100
validation_data = pd.DataFrame({
    'Method': ['Manual\n(Baseline)', 'Automated\n(Measured)'],
    'Pass Rate': [manual_val_rate, auto_val_rate]
})
bars = ax2.bar(validation_data['Method'], validation_data['Pass Rate'], 
               color=['lightcoral', 'lightgreen'], edgecolor='black', linewidth=1.5)
ax2.axhline(95, color='red', linestyle='--', linewidth=2, label='Target (95%)', alpha=0.7)
ax2.set_ylabel('Validation Pass Rate (%)', fontweight='bold')
ax2.set_title('B. Data Validation Accuracy', fontweight='bold', pad=10)
ax2.set_ylim([0, 105])
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels
for i, bar in enumerate(bars):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 1,
             f'{height:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

# Panel C: Performance thresholds (horizontal bar)
ax3 = fig.add_subplot(gs[1, 0])
p50_actual = df['latency_sec'].quantile(0.5)
p95_actual = df['latency_sec'].quantile(0.95)
thresholds = pd.DataFrame({
    'Metric': ['P50 Latency\n(≤30s)', 'P95 Latency\n(≤120s)', 'Validation\n(≥95%)'],
    'Target': [30, 120, 95],
    'Actual': [p50_actual, p95_actual, auto_val_rate]
})

y_pos = np.arange(len(thresholds))
ax3.barh(y_pos, thresholds['Target'], color='lightgray', alpha=0.5, 
         label='Target', edgecolor='black', linewidth=1)
ax3.barh(y_pos, thresholds['Actual'], 
         color=['green' if a <= t or (i == 2 and a >= t) else 'orange' 
                for i, (a, t) in enumerate(zip(thresholds['Actual'], thresholds['Target']))],
         label='Actual', alpha=0.8, edgecolor='black', linewidth=1.5)

ax3.set_yticks(y_pos)
ax3.set_yticklabels(thresholds['Metric'])
ax3.set_xlabel('Value', fontweight='bold')
ax3.set_title('C. Success Threshold Achievement', fontweight='bold', pad=10)
ax3.legend(loc='lower right')
ax3.grid(True, alpha=0.3, axis='x')

# Panel D: Sample distribution (pie chart)
ax4 = fig.add_subplot(gs[1, 1])
source_counts = df['data_source'].value_counts()
colors_pie = sns.color_palette("colorblind", len(source_counts))
wedges, texts, autotexts = ax4.pie(source_counts.values, labels=source_counts.index, 
                                     autopct='%1.1f%%', startangle=90, colors=colors_pie,
                                     textprops={'fontsize': 9, 'weight': 'bold'})
ax4.set_title(f'D. Data Source Distribution (n={len(df):,})', fontweight='bold', pad=10)

# Overall title
fig.suptitle('Serverless FHIR Pipeline Performance: Executive Summary', 
             fontsize=16, fontweight='bold', y=0.98)

# Save in multiple formats
plt.savefig(pub_dir / 'executive_summary.png', dpi=300, bbox_inches='tight')
plt.savefig(pub_dir / 'executive_summary.pdf', bbox_inches='tight')
plt.savefig(pub_dir / 'executive_summary.svg', bbox_inches='tight')
plt.show()

print("\n✓ Saved executive summary in PNG, PDF, and SVG formats")

## 4. Figure 2: Interactive Performance Dashboard (Plotly)

Interactive exploration of latency across dimensions.

In [ ]:
# Create interactive dashboard with multiple views
fig_interactive = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Latency Distribution by Data Source', 
                   'Latency vs File Size',
                   'Cumulative Distribution (CDF)',
                   'Performance Timeline'),
    specs=[[{'type': 'box'}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'scatter'}]]
)

# Panel 1: Box plot by data source
for source in df['data_source'].unique():
    source_data = df[df['data_source'] == source]['latency_sec']
    fig_interactive.add_trace(
        go.Box(y=source_data, name=source.title(), boxmean='sd'),
        row=1, col=1
    )

# Panel 2: Scatter plot (latency vs file size)
fig_interactive.add_trace(
    go.Scatter(
        x=df['file_size_kb'],
        y=df['latency_sec'],
        mode='markers',
        marker=dict(size=5, opacity=0.6, color=df['data_source'].astype('category').cat.codes,
                   colorscale='Viridis', showscale=True),
        text=df['data_source'],
        hovertemplate='<b>%{text}</b><br>File Size: %{x:.1f} KB<br>Latency: %{y:.2f}s<extra></extra>'
    ),
    row=1, col=2
)

# Panel 3: Cumulative distribution function
sorted_latency = np.sort(df['latency_sec'])
cdf = np.arange(1, len(sorted_latency) + 1) / len(sorted_latency) * 100

fig_interactive.add_trace(
    go.Scatter(
        x=sorted_latency,
        y=cdf,
        mode='lines',
        line=dict(color='steelblue', width=2),
        name='Automated',
        hovertemplate='Latency: %{x:.2f}s<br>Percentile: %{y:.1f}%<extra></extra>'
    ),
    row=2, col=1
)

# Add threshold lines to CDF
fig_interactive.add_vline(x=30, line_dash="dash", line_color="green", 
                         annotation_text="P50 Target (30s)", row=2, col=1)
fig_interactive.add_vline(x=120, line_dash="dash", line_color="orange", 
                         annotation_text="P95 Target (120s)", row=2, col=1)

# Panel 4: Performance over record index (simulated timeline)
# Add index for timeline visualization
df_sample = df.sample(min(1000, len(df)), random_state=12345).copy()
df_sample = df_sample.sort_values('latency_sec').reset_index(drop=True)

fig_interactive.add_trace(
    go.Scatter(
        x=df_sample.index,
        y=df_sample['latency_sec'],
        mode='markers',
        marker=dict(size=4, opacity=0.6, color=df_sample['latency_sec'],
                   colorscale='RdYlGn_r', showscale=False),
        text=df_sample['data_source'],
        hovertemplate='Record: %{x}<br>Latency: %{y:.2f}s<br>Source: %{text}<extra></extra>'
    ),
    row=2, col=2
)

# Add mean line
mean_latency = df_sample['latency_sec'].mean()
fig_interactive.add_hline(y=mean_latency, line_dash="dash", line_color="red",
                         annotation_text=f"Mean: {mean_latency:.2f}s", row=2, col=2)

# Update layout
fig_interactive.update_xaxes(title_text="Data Source", row=1, col=1)
fig_interactive.update_xaxes(title_text="File Size (KB)", row=1, col=2)
fig_interactive.update_xaxes(title_text="Latency (seconds)", row=2, col=1)
fig_interactive.update_xaxes(title_text="Record Index (sorted by latency)", row=2, col=2)

fig_interactive.update_yaxes(title_text="Latency (seconds)", row=1, col=1)
fig_interactive.update_yaxes(title_text="Latency (seconds)", row=1, col=2)
fig_interactive.update_yaxes(title_text="Cumulative Percentage", row=2, col=1)
fig_interactive.update_yaxes(title_text="Latency (seconds)", row=2, col=2)

fig_interactive.update_layout(
    title_text="Interactive Performance Analysis Dashboard",
    title_font_size=18,
    showlegend=False,
    height=800,
    hovermode='closest'
)

# Save interactive HTML
fig_interactive.write_html(interactive_dir / 'performance_dashboard.html')
fig_interactive.show()

print("\n✓ Saved interactive dashboard as HTML")

## 5. Figure 3: Statistical Results Visualization (Matplotlib)

Publication-quality figure showing hypothesis test results.

In [ ]:
# Create statistical results figure
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel A: Effect size visualization
ax1 = axes[0]
cohens_d = results.get('cohens_d', 0)
effect_categories = ['Negligible\n(d<0.2)', 'Small\n(0.2≤d<0.5)', 
                     'Medium\n(0.5≤d<0.8)', 'Large\n(d≥0.8)']
effect_boundaries = [0, 0.2, 0.5, 0.8, max(1.5, cohens_d + 0.2)]

# Create horizontal bar showing effect size scale
for i in range(len(effect_categories)):
    width = effect_boundaries[i+1] - effect_boundaries[i]
    color = 'lightcoral' if i < 2 else 'lightyellow' if i == 2 else 'lightgreen'
    ax1.barh(0, width, left=effect_boundaries[i], height=0.5, 
             color=color, edgecolor='black', linewidth=1)
    
    # Add category labels
    mid_point = (effect_boundaries[i] + effect_boundaries[i+1]) / 2
    ax1.text(mid_point, 0, effect_categories[i], 
             ha='center', va='center', fontsize=9, fontweight='bold')

# Add actual Cohen's d marker
ax1.scatter([cohens_d], [0], s=300, c='red', marker='v', 
           edgecolors='black', linewidth=2, zorder=5, label=f'Observed\n(d={cohens_d:.2f})')
ax1.set_ylim([-0.5, 0.5])
ax1.set_xlim([0, effect_boundaries[-1]])
ax1.set_xlabel('Cohen\'s d', fontweight='bold')
ax1.set_title('A. Effect Size Interpretation', fontweight='bold', pad=10)
ax1.set_yticks([])
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3, axis='x')

# Panel B: P-value and significance
ax2 = axes[1]
p_value = results.get('paired_ttest_pvalue_one_tail', 0.5)
p_value_display = max(p_value, 1e-6)  # For log scale visualization

# Create significance visualization
significance_levels = [0.001, 0.01, 0.05, 0.1, 1.0]
sig_labels = ['***\n(p<0.001)', '**\n(p<0.01)', '*\n(p<0.05)', 
              'n.s.\n(p<0.1)', 'n.s.\n(p≥0.1)']
sig_colors = ['darkgreen', 'green', 'lightgreen', 'lightyellow', 'lightcoral']

y_pos = 0
for i in range(len(sig_labels)):
    left = significance_levels[i-1] if i > 0 else 1e-6
    width = significance_levels[i] - left
    ax2.barh(y_pos, width, left=left, height=0.5, 
             color=sig_colors[i], edgecolor='black', linewidth=1, alpha=0.7)
    
    mid_point = np.sqrt(left * significance_levels[i])  # Geometric mean for log scale
    ax2.text(mid_point, y_pos, sig_labels[i], 
             ha='center', va='center', fontsize=9, fontweight='bold')

# Add actual p-value marker
ax2.scatter([p_value_display], [y_pos], s=300, c='red', marker='v', 
           edgecolors='black', linewidth=2, zorder=5, 
           label=f'Observed\n(p={p_value:.4f})')
ax2.set_xscale('log')
ax2.set_xlim([1e-6, 1])
ax2.set_ylim([-0.5, 0.5])
ax2.set_xlabel('P-value (log scale)', fontweight='bold')
ax2.set_title('B. Statistical Significance', fontweight='bold', pad=10)
ax2.set_yticks([])
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3, axis='x')

# Panel C: Confidence interval
ax3 = axes[2]
mean_improvement = results.get('mean_improvement_sec', 0)
# Calculate CI if not in results (approximate)
df['latency_diff'] = results.get('manual_baseline_sec', 1200) - df['latency_sec']
ci_lower = mean_improvement - 1.96 * (df['latency_diff'].std() / np.sqrt(len(df)))
ci_upper = mean_improvement + 1.96 * (df['latency_diff'].std() / np.sqrt(len(df)))

ax3.errorbar([1], [mean_improvement], 
             yerr=[[mean_improvement - ci_lower], [ci_upper - mean_improvement]],
             fmt='o', markersize=15, capsize=10, capthick=2, linewidth=2,
             color='steelblue', markeredgecolor='black', markeredgewidth=2)
ax3.axhline(0, color='red', linestyle='--', linewidth=2, label='No Difference', alpha=0.7)
ax3.set_xlim([0.5, 1.5])
ax3.set_ylabel('Mean Improvement (seconds)', fontweight='bold')
ax3.set_title('C. 95% Confidence Interval', fontweight='bold', pad=10)
ax3.set_xticks([1])
ax3.set_xticklabels(['Latency\nImprovement'])
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# Add annotation
ax3.text(1, mean_improvement + (ci_upper - ci_lower)/2 + 50, 
         f'Mean: {mean_improvement:.1f}s\n95% CI: [{ci_lower:.1f}, {ci_upper:.1f}]',
         ha='center', va='bottom', fontsize=10, fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

fig.suptitle('Statistical Hypothesis Testing Results', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()

# Save in multiple formats
plt.savefig(pub_dir / 'statistical_results.png', dpi=300, bbox_inches='tight')
plt.savefig(pub_dir / 'statistical_results.pdf', bbox_inches='tight')
plt.savefig(pub_dir / 'statistical_results.svg', bbox_inches='tight')
plt.show()

print("\n✓ Saved statistical results figure in PNG, PDF, and SVG formats")

## 6. Figure 4: Interactive Threshold Achievement (Plotly)

Gauge charts showing success criteria achievement.

In [ ]:
# Create gauge dashboard for threshold achievement
fig_gauges = make_subplots(
    rows=1, cols=3,
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}]],
    subplot_titles=('P50 Latency Target', 'P95 Latency Target', 'Validation Accuracy Target')
)

# Gauge 1: P50 Latency (target ≤30s)
p50_actual = df['latency_sec'].quantile(0.5)
p50_pct = min(100, (30 / p50_actual) * 100) if p50_actual > 0 else 100

fig_gauges.add_trace(go.Indicator(
    mode="gauge+number+delta",
    value=p50_actual,
    domain={'x': [0, 1], 'y': [0, 1]},
    title={'text': f"P50: {p50_actual:.2f}s", 'font': {'size': 16}},
    delta={'reference': 30, 'increasing': {'color': "red"}, 'decreasing': {'color': "green"}},
    gauge={
        'axis': {'range': [None, 60], 'tickwidth': 1, 'tickcolor': "darkblue"},
        'bar': {'color': "darkblue"},
        'bgcolor': "white",
        'borderwidth': 2,
        'bordercolor': "gray",
        'steps': [
            {'range': [0, 30], 'color': 'lightgreen'},
            {'range': [30, 60], 'color': 'lightcoral'}],
        'threshold': {
            'line': {'color': "red", 'width': 4},
            'thickness': 0.75,
            'value': 30}
    }
), row=1, col=1)

# Gauge 2: P95 Latency (target ≤120s)
p95_actual = df['latency_sec'].quantile(0.95)

fig_gauges.add_trace(go.Indicator(
    mode="gauge+number+delta",
    value=p95_actual,
    domain={'x': [0, 1], 'y': [0, 1]},
    title={'text': f"P95: {p95_actual:.2f}s", 'font': {'size': 16}},
    delta={'reference': 120, 'increasing': {'color': "red"}, 'decreasing': {'color': "green"}},
    gauge={
        'axis': {'range': [None, 200], 'tickwidth': 1, 'tickcolor': "darkblue"},
        'bar': {'color': "darkblue"},
        'bgcolor': "white",
        'borderwidth': 2,
        'bordercolor': "gray",
        'steps': [
            {'range': [0, 120], 'color': 'lightgreen'},
            {'range': [120, 200], 'color': 'lightcoral'}],
        'threshold': {
            'line': {'color': "red", 'width': 4},
            'thickness': 0.75,
            'value': 120}
    }
), row=1, col=2)

# Gauge 3: Validation Accuracy (target ≥95%)
val_actual = df['validation_pass_bool'].mean() * 100

fig_gauges.add_trace(go.Indicator(
    mode="gauge+number+delta",
    value=val_actual,
    domain={'x': [0, 1], 'y': [0, 1]},
    title={'text': f"Validation: {val_actual:.1f}%", 'font': {'size': 16}},
    delta={'reference': 95, 'increasing': {'color': "green"}, 'decreasing': {'color': "red"}},
    gauge={
        'axis': {'range': [0, 100], 'tickwidth': 1, 'tickcolor': "darkblue"},
        'bar': {'color': "darkblue"},
        'bgcolor': "white",
        'borderwidth': 2,
        'bordercolor': "gray",
        'steps': [
            {'range': [0, 95], 'color': 'lightcoral'},
            {'range': [95, 100], 'color': 'lightgreen'}],
        'threshold': {
            'line': {'color': "green", 'width': 4},
            'thickness': 0.75,
            'value': 95}
    }
), row=1, col=3)

fig_gauges.update_layout(
    title_text="Success Criteria Achievement Dashboard",
    title_font_size=20,
    height=400
)

# Save interactive HTML
fig_gauges.write_html(interactive_dir / 'threshold_gauges.html')
fig_gauges.show()

print("\n✓ Saved threshold gauge dashboard as HTML")

## 7. Figure 5: Cost-Benefit Analysis (Matplotlib)

Visualize time savings and cost implications.

In [ ]:
# Create cost-benefit visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Time savings per record
ax1 = axes[0]
manual_time_min = results.get('manual_baseline_min', 20)
auto_time_min = results.get('automated_mean_sec', 30) / 60
time_saved_min = manual_time_min - auto_time_min

categories = ['Manual\nBaseline', 'Automated\nPipeline', 'Time\nSaved']
times = [manual_time_min, auto_time_min, time_saved_min]
colors_bar = ['lightcoral', 'lightgreen', 'gold']

bars = ax1.bar(categories, times, color=colors_bar, edgecolor='black', linewidth=2)
ax1.set_ylabel('Time per Record (minutes)', fontweight='bold', fontsize=12)
ax1.set_title('A. Time Savings per Record', fontweight='bold', fontsize=14, pad=10)
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.5,
             f'{height:.1f} min', ha='center', va='bottom', 
             fontweight='bold', fontsize=11)

# Panel B: Projected annual savings
ax2 = axes[1]
records_per_year = [100, 500, 1000, 5000, 10000]
hours_saved = [(r * time_saved_min) / 60 for r in records_per_year]

ax2.plot(records_per_year, hours_saved, marker='o', linewidth=3, 
         markersize=10, color='steelblue', markeredgecolor='black', 
         markeredgewidth=2)
ax2.fill_between(records_per_year, 0, hours_saved, alpha=0.3, color='steelblue')
ax2.set_xlabel('Records Processed per Year', fontweight='bold', fontsize=12)
ax2.set_ylabel('Time Saved (hours/year)', fontweight='bold', fontsize=12)
ax2.set_title('B. Projected Annual Time Savings', fontweight='bold', fontsize=14, pad=10)
ax2.grid(True, alpha=0.3)

# Add annotations for key points
for x, y in zip(records_per_year[::2], hours_saved[::2]):
    ax2.annotate(f'{y:.0f}h', xy=(x, y), xytext=(10, 10), 
                textcoords='offset points', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

fig.suptitle('Time Savings Analysis: Manual vs Automated Processing', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()

# Save in multiple formats
plt.savefig(pub_dir / 'cost_benefit_analysis.png', dpi=300, bbox_inches='tight')
plt.savefig(pub_dir / 'cost_benefit_analysis.pdf', bbox_inches='tight')
plt.savefig(pub_dir / 'cost_benefit_analysis.svg', bbox_inches='tight')
plt.show()

print("\n✓ Saved cost-benefit analysis in PNG, PDF, and SVG formats")

## 8. Summary: All Figures Generated

In [ ]:
# List all generated figures
print("\n" + "="*70)
print("VISUALIZATION DASHBOARD SUMMARY")
print("="*70)

print("\n📊 PUBLICATION FIGURES (Static, High-Resolution)")
print("-" * 70)
pub_figures = sorted(pub_dir.glob('*'))
for i, fig_file in enumerate(pub_figures, 1):
    print(f"  {i}. {fig_file.name}")

print(f"\nTotal publication figures: {len(pub_figures)}")
print(f"Location: {pub_dir}")

print("\n🌐 INTERACTIVE DASHBOARDS (HTML)")
print("-" * 70)
interactive_figures = sorted(interactive_dir.glob('*.html'))
for i, fig_file in enumerate(interactive_figures, 1):
    print(f"  {i}. {fig_file.name}")

print(f"\nTotal interactive dashboards: {len(interactive_figures)}")
print(f"Location: {interactive_dir}")

print("\n✓ All visualizations generated successfully!")
print("\nNext Steps:")
print("  1. Review publication figures for report inclusion")
print("  2. Open interactive dashboards in web browser for exploration")
print("  3. Include figures in Phase 8 results interpretation")
print("  4. Prepare figure captions and references for final report")
print("="*70)